# 🟦 01 - Setup

## 🔹 Paths and Configuration

In [0]:
from pyspark.sql.functions import lit, current_timestamp, input_file_name,col, regexp_extract
from pathlib import Path
import unicodedata
import unicodedata
import re
from datetime import datetime


catalog = "workspace"
schema = "default"
volume = "raw_files"

RAW_BASE = f"/Volumes/{catalog}/{schema}/{volume}/fuel_dashboard/raw"
BRONZE_BASE = f"/Volumes/{catalog}/{schema}/{volume}/fuel_dashboard/bronze"

run_date = datetime.today().strftime("%Y-%m-%d")

SOURCES = [
    "anp_prices",
    "anp_refinery",
    "anp_imports",
    "brent"
]

# 🟦 02 - Functions


In [0]:
def normalize_column(col_name):
    col = unicodedata.normalize("NFKD", col_name)
    col = col.encode("ASCII", "ignore").decode("ASCII")
    col = col.lower().strip()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    return col

#clean columns
def clean_columns(df):
    return df.toDF(*[normalize_column(c) for c in df.columns])

#List and ignore empty archives
def list_valid_files(folder_path):
    files = dbutils.fs.ls(folder_path)
    return [f.path for f in files if f.size > 0 and f.path.endswith(".csv")]

#choose enconding
def get_read_options(source_name):
    if source_name in ["anp_prices"]:
        return {
            "encoding": "latin1",
            "sep": ";"
        }
    else:
        return {
            "encoding": "utf-8",
            "sep": ","
        }

In [0]:
def process_bronze_source(source_name):
    print(f"\nProcessing: {source_name}")

    raw_path = f"{RAW_BASE}/{source_name}/run_date={run_date}"
    output_path = f"{BRONZE_BASE}/{source_name}"
    table_name = f"workspace.default.bronze_{source_name}"

    files = list_valid_files(raw_path)

    if not files:
        print("No valid files found.")
        return

    options = get_read_options(source_name)

    df = spark.read \
        .option("header", True) \
        .option("encoding", options["encoding"]) \
        .option("sep", options["sep"]) \
        .csv(files)

    df = clean_columns(df)

    df = df \
        .withColumn("source_name", lit(source_name)) \
        .withColumn("ingested_at", current_timestamp()) \
        .withColumn("run_date", lit(run_date)) \
        .withColumn("source_file", col("_metadata.file_path")) \
        .withColumn("source_file_name", regexp_extract(col("_metadata.file_path"), r"([^/]+$)", 1))

    print("Rows:", df.count())
    print("Columns:", len(df.columns))



    df.write \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(table_name)

    print(f"Saved to {output_path}")

# 🟦 03 - Execution

In [0]:
for source in SOURCES:
    process_bronze_source(source)

# 🟦 04 - Validation



In [0]:
for source in SOURCES:
    path = f"{BRONZE_BASE}/{source}"
    
    try:
        df = spark.read.format("delta").load(path)
        print(f"\n{source}")
        print("Rows:", df.count())
        display(df.limit(5))
    except Exception as e:
        print(f"{source} ERROR:", e)

In [0]:
for source in SOURCES:
    path = f"{BRONZE_BASE}/{source}"
    try:
        df = spark.read.format("delta").load(path)
        print(f"\n{source}")
        print("columns:", df.columns)
        print("schema:", df.printSchema())
        display(df.limit(5))
    except Exception as e:
        print(f"{source} ERROR:", e)



In [0]:
SOURCES[0]

In [0]:
path = f"{BRONZE_BASE}/{SOURCES[0]}"

df_PRICES = spark.read.format("delta").load(path)

In [0]:
display(df_PRICES)